# CropMS-LLM ViT — full pipeline notebook

End-to-end walkthrough of the shipped model:
1. **Inference on a sample tile** — same path the React/FastAPI UI takes.
2. **Eval readout** — held-out test metrics (3-class + 6-class mIoU, NDVI/NDRE MAE).
3. **Training** — full retrain command + an optional smoke-test cell.

Runs on **CPU or CUDA**. The shipped checkpoint `runs/cropmsllm_vit.pt` is loaded
with `weights_only=False` (PyTorch 2.3 compatible).

## 0. Setup

If you launched Jupyter from the project root with the project venv active, no path tweaks needed.

In [ ]:
from __future__ import annotations
import os, sys, json, time
from pathlib import Path

# Make the project root importable when this notebook lives in notebooks/
REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np
import torch
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'project root  : {REPO}')
print(f'torch         : {torch.__version__}')
print(f'device        : {DEVICE}')
if DEVICE == 'cuda':
    print(f'cuda device   : {torch.cuda.get_device_name(0)}')
    print(f'vram (GB)     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}')

In [ ]:
CKPT = REPO / 'runs' / 'cropmsllm_vit.pt'
DATA_ROOT = REPO / 'data' / 'weedsgalore' / 'weedsgalore-dataset'
EVAL_METRICS = REPO / 'runs' / 'eval_metrics_extended.json'

for p in (CKPT, DATA_ROOT, EVAL_METRICS):
    print(('OK ' if p.exists() else 'MISS'), p)

## 1. Inference on a sample tile (UI-equivalent)

Five tiles ship under `data/weedsgalore/weedsgalore-dataset/`, spanning the NDVI range
from heavily stressed bare soil to relatively healthy crop. Pick one, load the 5-band
MS stack, run `V3Predictor.predict()`, then visualize the same 5 outputs the UI shows.

In [ ]:
BUNDLED_TILES = [
    '2023-06-06_0166',  # NDVI=-0.149  most stressed / bare
    '2023-06-15_0169',  # NDVI=-0.068
    '2023-05-30_0044',  # NDVI=-0.023
    '2023-05-25_0183',  # NDVI=-0.002
    '2023-05-25_0109',  # NDVI=+0.038  relatively healthy
]

TILE_ID = BUNDLED_TILES[0]                 # ← change to try other tiles
DATE = TILE_ID.rsplit('_', 1)[0]
TILE_R = DATA_ROOT / DATE / 'images' / f'{TILE_ID}_R.png'
print('tile:', TILE_R)
assert TILE_R.exists(), f'missing tile band: {TILE_R}'

In [ ]:
from src.inference.pipeline import load_ms_tile
from src.inference.pipeline_v3 import V3Predictor

ms, _crs, _transform = load_ms_tile(TILE_R)
print('ms shape:', ms.shape, '| dtype:', ms.dtype, '| range:', float(ms.min()), '..', float(ms.max()))

predictor = V3Predictor(CKPT, device=DEVICE, use_tta=True)
print('encoder      :', predictor.encoder)
print('num_classes  :', predictor.num_classes)
print('train HxW    :', predictor.train_hw)

t0 = time.perf_counter()
out = predictor.predict(ms)
print(f'inference    : {time.perf_counter() - t0:.2f}s on {DEVICE}')
print('seg shape    :', out['seg'].shape, '| classes used:', sorted(np.unique(out['seg']).tolist()))
print('ndvi range   : %.3f .. %.3f  (mean %.3f)' % (out['ndvi_pred'].min(), out['ndvi_pred'].max(), out['ndvi_pred'].mean()))
print('ndre range   : %.3f .. %.3f  (mean %.3f)' % (out['ndre_pred'].min(), out['ndre_pred'].max(), out['ndre_pred'].mean()))
print('canopy frac  : %.3f' % out['canopy_pred'])

In [ ]:
# Reuse the API's visualization helpers — same pixels the UI shows.
from api.main import colorize_seg, rgb_composite
from src.inference.task_outputs import compute_all, colorize_stress, colorize_zones

rgb = rgb_composite(ms)
seg_rgb = colorize_seg(out['seg'], num_classes=predictor.num_classes)
tasks = compute_all(ms, out['seg'], out['ndvi_pred'], out['ndre_pred'])
stress_rgb = colorize_stress(tasks.stress_type_map)
zone_rgb = colorize_zones(tasks.field_zone_map)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
panels = [
    ('RGB composite (R/G/B, p2-p98)', rgb, None),
    ('Seg mask (6-class)', seg_rgb, None),
    ('NDVI (predicted)', out['ndvi_pred'], 'RdYlGn'),
    ('NDRE (predicted)', out['ndre_pred'], 'RdYlGn'),
    (f'Stress type (dominant: {tasks.stress_type})', stress_rgb, None),
    ('Field zoning', zone_rgb, None),
]
for ax, (title, img, cmap) in zip(axes.flat, panels):
    if cmap is None:
        ax.imshow(img)
    else:
        im = ax.imshow(img, cmap=cmap, vmin=-0.2, vmax=0.8)
        plt.colorbar(im, ax=ax, fraction=0.045, pad=0.02)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
fig.suptitle(f'{TILE_ID}', fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# Structured outputs — these are exactly what /api/predict returns to the UI.
print('crop health        :', tasks.crop_health, f'(NDVI score {tasks.crop_health_score:+.3f})')
print('dominant stress    :', tasks.stress_type)
print('NDVI summary       :', tasks.ndvi_summary)
print('field zone summary :', tasks.field_zone_summary)
print()
print('spatial 3x3 readout:')
print(tasks.spatial_text)

### Try the other bundled tiles

Edit `TILE_ID` above and re-run the inference cells. The 5 tiles are picked to span
the operating range — useful for checking the model's behavior on stressed vs healthy
crop without needing the full WeedsGalore dataset (~27 GB).

## 2. Eval readout

The shipped checkpoint was evaluated on the WACV 2025 WeedsGalore test split (26 tiles).
These metrics are baked into `runs/eval_metrics_extended.json` and surfaced by `/api/state`.

In [ ]:
metrics = json.loads(EVAL_METRICS.read_text())
test = metrics['splits']['test']
headline = [
    ('3-class mIoU (bg/crop/weed)',  test['mean_iou_3class']),
    ('6-class mIoU',                 test.get('mean_iou_6class', test.get('mean_iou'))),
    ('Pixel accuracy',               test['pixel_accuracy']),
    ('NDVI MAE',                     test['ndvi_mae']),
    ('NDRE MAE',                     test['ndre_mae']),
]
for k, v in headline:
    print(f'{k:32s}  {v:.4f}')
print()
if 'per_class_iou' in test:
    print('per-class IoU:')
    for name, iou in test['per_class_iou'].items():
        print(f'  {name:20s}  {iou:.4f}')

In [ ]:
# Re-run eval against the bundled subset (5 tiles in splits/test.txt).
# Numbers will differ from the headline because the bundled tiles are a small slice.
!python {REPO}/scripts/eval.py --splits test

## 3. Training

Full training takes **~25 min on an RTX 4060 8 GB**, longer on smaller GPUs and *much*
longer on CPU (recommended only for a smoke test). The recipe is in `scripts/train.py`:

- MiT-B3 SegFormer encoder, DeepLabv3+ decoder, 4 heads (seg + NDVI + NDRE + canopy)
- 5 warmup epochs (encoder frozen) + 35 main epochs (full unfreeze, cosine LR)
- Augment: H/V flip + rot90 + weed copy-paste + MS-aware mixup
- Kendall multi-task uncertainty loss
- Mixed precision (fp16 autocast)

**To retrain on the full WeedsGalore dataset (you need to download it separately
— the ship only includes 5 sample tiles):**

```bash
# from the project root, with the venv active
python scripts/train.py
```

This writes a fresh checkpoint to `runs/cropmsllm_vit.pt` and a training log to
`runs/train_<encoder>.log.json`.

In [ ]:
# Optional smoke-test training cell — DISABLED by default.
# Set RUN_SMOKE_TRAIN = True to launch a 1-epoch run on whatever splits/train.txt
# points at. With the bundled subset this needs you to add tile ids to
# splits/train.txt; otherwise the dataset will be empty and training will exit.
RUN_SMOKE_TRAIN = False

if RUN_SMOKE_TRAIN:
    !python {REPO}/scripts/train.py --epochs-warmup 0 --epochs-main 1 --batch-size 2
else:
    print('Smoke train disabled. Flip RUN_SMOKE_TRAIN to True to launch.')